In [5]:
import pickle
import networkx as nx
import pennylane as qml
from joblib import Parallel, delayed
from pennylane import numpy as np
from itertools import combinations



In [20]:
def load_P_w(idx,graph_type):
    #loads incidence matrix and weights
    if idx < 10:
        hypergraph_name = f"0{idx}"
    else:
        hypergraph_name = f"{idx}"
    print("Name: ", hypergraph_name)
    P = np.load(f'data/{graph_type}/P/{hypergraph_name}.npy')
    w = np.load(f'data/{graph_type}/w/{hypergraph_name}.npy')
    return P,w
# def make_M_V(P,w):
#     #return imbalance and variance matrices
#     M = P @ np.diag(w) @ P.T
#     return M, np.diag(P @ w) - M
def make_Vlist(P,w):
    #weighted variance matrix for eeach hyperedge
    Vlist=[]
    for col in range(P.shape[1]):
        p=P[:,col]
        #V_e = diag(p_e) -M_e --> diag(p_e) * 
        V_e = w[col]*(np.diag(p)-np.outer(p,p))
        Vlist.append(V_e)
    return Vlist
    

In [21]:
P = np.array([
    [1/3, 0.0],
    [1/3, 0.25],
    [0.0,     0.25],
    [0.0,     0.25],
    [1/3, 0.25]
], requires_grad=True)

w = np.ones(2)
Vlist=make_Vlist(P,w)
print(Vlist[0])

[[ 0.22222222 -0.11111111  0.          0.         -0.11111111]
 [-0.11111111  0.22222222  0.          0.         -0.11111111]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [-0.11111111 -0.11111111  0.          0.          0.22222222]]


In [22]:
print(P.shape)

(5, 2)


In [61]:
def make_dicke_state(n, k):
        if k < 0 or k > n:
            raise ValueError("k must satisfy 0 <= k <= n")
    
        state = np.zeros(2**n, dtype=complex)
        weight_k_states = list(combinations(range(n), k))
        
        amplitude = 1 / np.sqrt(len(weight_k_states))

        #convert bit string to integer
        for one_positions in weight_k_states:
            basis_index=0
            for wire in one_positions:
                basis_index+=2**(n-1-wire)
            state[basis_index] = amplitude

        

        d_state = np.array(state, requires_grad=False)
        if (comb(n,k)!=np.count_nonzero(d_state)):
            raise ValueError("amplitude count incorrect")
    
        return d_state

In [62]:
def fair_cut_qaoa(Vlist,
                  P,
                  w,
                  seed:int,
                  num_layers:int,
                  scaling_constant:float,
                  step_size:float,
                  max_trials:float,
                 ):
    #List of hyperedges
    rng= np.random.default_rng(seed=seed)
    hyper_edges_list = []
    for hyper_edge_matrix in Vlist:
        hyper_edge = []
        hyper_edge = set(np.where(hyper_edge_matrix != 0.0)[1])
        hyper_edges_list.append(hyper_edge)
        
    #Clique expansion for each hyperedge
    n_wires = Vlist[0].shape[0]
    edge_list = [] #list of all edges after clique expansion
    hyper_edges_clique_enum = [] # stores lists where each list is the clique expansion of a hyperedge
    hyper_edge_idx = 0
    for hyper_edge in hyper_edges_list:
        hyper_edge_expansion = []
        for edge in combinations(hyper_edge, 2):
            hyper_edge_expansion.append((np.min(edge), np.max(edge), float(Vlist[hyper_edge_idx][np.min(edge)][np.max(edge)])))
        edge_list += hyper_edge_expansion
        hyper_edges_clique_enum.append(hyper_edge_expansion)
        hyper_edge_idx += 1
    #print("Len Edge list: ", len(edge_list))
    edge_list = set(edge_list)
    #print("Len Set Edge list: ", len(edge_list))
    num_edges = len(edge_list)

    #initialize parameters
    beta_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)
    gamma_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)

    def U_B(beta):
        edge_idx=0
        for u, v, _ in edge_list:
            theta = beta[edge_idx] 
            qml.S(wires=u) 
            qml.SingleExcitationMinus(-2*theta,wires=[u,v])
            qml.adjoint(qml.S)(wires=u) 
            edge_idx+=1

    
    
    

In [59]:
def qaoa_simulation(idx,graph_type,num_layers:int,P=None,w=None):
    if idx==-1:
        P,w = P,w
    else:
        P,w= load_P_w(idx,graph_type)
    Vlist = make_Vlist(P,w)
    fair_cut_qaoa(Vlist,P,w,0,0,0.0,0.0,0.0)
    
    

In [60]:
qaoa_simulation(-1,"temp",1,P=P,w=w)

[[(np.int64(0), np.int64(1), -0.1111111111111111), (np.int64(0), np.int64(4), -0.1111111111111111), (np.int64(1), np.int64(4), -0.1111111111111111)], [(np.int64(1), np.int64(2), -0.0625), (np.int64(1), np.int64(3), -0.0625), (np.int64(1), np.int64(4), -0.0625), (np.int64(2), np.int64(3), -0.0625), (np.int64(2), np.int64(4), -0.0625), (np.int64(3), np.int64(4), -0.0625)]]
